In [35]:
import sqlite3
import pandas as pd

In [36]:
# Load the CSV file into a pandas DataFrame, specifying encoding to handle potential issues
df = pd.read_csv('/content/Sample - Superstore.csv', encoding='latin1')
display(df.head())

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [37]:
# Create an in-memory SQLite database connection
conn = sqlite3.connect(':memory:')

# Write the DataFrame to a SQL table named 'superstore_raw'
# if_exists='replace' will overwrite the table if it already exists
df.to_sql('superstore_raw', conn, if_exists='replace', index=False)

print("Data successfully loaded into 'superstore_raw' table.")

# Verify by listing tables in the database
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print("\nTables in the database:", tables)

Data successfully loaded into 'superstore_raw' table.

Tables in the database: [('superstore_raw',)]


### Create `customers` table

This table will store unique customer information.

In [38]:
conn.execute("DROP TABLE IF EXISTS customers;")
create_customers_query = """
CREATE TABLE customers AS
SELECT
    "Customer ID" AS customer_id,
    "Customer Name" AS customer_name,
    MIN(Segment) AS segment,
    MIN(Country) AS country,
    MIN(City) AS city,
    MIN(State) AS state,
    MIN("Postal Code") AS postal_code,
    MIN(Region) AS region
FROM superstore_raw
GROUP BY "Customer ID", "Customer Name";
"""
conn.execute(create_customers_query)

print("Customers table created.")

# Verify by selecting a few rows
customers_df = pd.read_sql_query("SELECT * FROM customers LIMIT 5", conn)
display(customers_df)

ProgrammingError: You can only execute one statement at a time.

### Create `products` table

This table will store unique product information.

In [ ]:
create_products_query = """
CREATE TABLE products AS
SELECT DISTINCT
    "Product ID" AS product_id,
    Category AS category,
    "Sub-Category" AS sub_category,
    "Product Name" AS product_name
FROM superstore_raw;
"""
conn.execute(create_products_query)

print("Products table created.")

# Verify by selecting a few rows
products_df = pd.read_sql_query("SELECT * FROM products LIMIT 5", conn)
display(products_df)

### Create `orders` table

This table will store unique order information, linking to customers.

In [ ]:
create_orders_query = """
CREATE TABLE orders AS
SELECT DISTINCT
    "Order ID" AS order_id,
    "Order Date" AS order_date,
    "Ship Date" AS ship_date,
    "Ship Mode" AS ship_mode,
    "Customer ID" AS customer_id
FROM superstore_raw;
"""
conn.execute(create_orders_query)

print("Orders table created.")

# Verify by selecting a few rows
orders_df = pd.read_sql_query("SELECT * FROM orders LIMIT 5", conn)
display(orders_df)

### Verify all new tables

Let's check all tables currently in the database.

In [ ]:
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
all_tables = cursor.fetchall()
print("\nAll tables in the database:", all_tables)

### Orders with Sales Greater Than Average Sales

This query will select all orders from the `superstore_raw` table where the `Sales` value for that order is greater than the overall average sales across all orders. A subquery is used to calculate the average sales first.

In [ ]:
query_avg_sales = """
SELECT *
FROM superstore_raw
WHERE Sales > (
    SELECT AVG(Sales)
    FROM superstore_raw
);
"""

avg_sales_orders_df = pd.read_sql_query(query_avg_sales, conn)
display(avg_sales_orders_df.head())
print(f"Total orders with sales greater than average: {len(avg_sales_orders_df)}")

### Highest Sales Order for Each Customer

This query will find the order with the highest sales for each unique customer. It uses a correlated subquery to compare each order's sales to the maximum sales for that specific customer.

In [ ]:
query_highest_sales_per_customer = """
SELECT s."Customer ID", s."Order ID", s.Sales
FROM superstore_raw s
WHERE s.Sales = (
    SELECT MAX(sub.Sales)
    FROM superstore_raw sub
    WHERE sub."Customer ID" = s."Customer ID"
)
ORDER BY s."Customer ID", s.Sales DESC;
"""

highest_sales_per_customer_df = pd.read_sql_query(query_highest_sales_per_customer, conn)
display(highest_sales_per_customer_df)
print(f"Total unique customers with their highest sales order: {len(highest_sales_per_customer_df)}")

### Total Sales for Each Customer (using CTE)

This query uses a Common Table Expression (CTE) to calculate the sum of `Sales` for each unique customer from the `superstore_raw` table.

In [ ]:
query_total_sales_per_customer_cte = """
WITH CustomerTotalSales AS (
    SELECT
        "Customer ID" AS customer_id,
        SUM(Sales) AS total_sales
    FROM superstore_raw
    GROUP BY "Customer ID"
)
SELECT
    customer_id,
    total_sales
FROM CustomerTotalSales
ORDER BY total_sales DESC;
"""

total_sales_per_customer_df = pd.read_sql_query(query_total_sales_per_customer_cte, conn)
display(total_sales_per_customer_df)
print(f"Total unique customers in this report: {len(total_sales_per_customer_df)}")

### Customers with Total Sales Above Average (CTE + Subquery)

This query identifies customers whose individual total sales are greater than the average total sales across all customers. It uses a CTE to first calculate each customer's total sales and then a subquery to determine the overall average of these totals.

In [ ]:
query_customers_above_avg_sales = """
WITH CustomerTotalSales AS (
    SELECT
        "Customer ID" AS customer_id,
        SUM(Sales) AS total_sales
    FROM superstore_raw
    GROUP BY "Customer ID"
)
SELECT
    customer_id,
    total_sales
FROM CustomerTotalSales
WHERE total_sales > (
    SELECT AVG(total_sales)
    FROM CustomerTotalSales
)
ORDER BY total_sales DESC;
"""

customers_above_avg_sales_df = pd.read_sql_query(query_customers_above_avg_sales, conn)
display(customers_above_avg_sales_df)
print(f"Number of customers with total sales above average: {len(customers_above_avg_sales_df)}")

### Rank Customers by Total Sales (Window Function)

This query uses a Common Table Expression (CTE) to calculate the total sales for each customer and then applies the `RANK()` window function to assign a rank based on these total sales, from highest to lowest.

In [ ]:
query_ranked_customers = """
WITH CustomerTotalSales AS (
    SELECT
        "Customer ID" AS customer_id,
        SUM(Sales) AS total_sales
    FROM superstore_raw
    GROUP BY "Customer ID"
)
SELECT
    customer_id,
    total_sales,
    RANK() OVER (ORDER BY total_sales DESC) AS sales_rank
FROM CustomerTotalSales
ORDER BY sales_rank;
"""

ranked_customers_df = pd.read_sql_query(query_ranked_customers, conn)
display(ranked_customers_df)
print(f"Total unique customers ranked: {len(ranked_customers_df)}")

### Assign Row Numbers to Each Order Within a Customer (Window Function + PARTITION BY)

This query uses the `ROW_NUMBER()` window function partitioned by `Customer ID` and ordered by `Order Date` to assign a unique sequential number to each order placed by a particular customer.

In [ ]:
query_row_number_per_customer = """
SELECT
    "Customer ID" AS customer_id,
    "Order ID" AS order_id,
    "Order Date" AS order_date,
    ROW_NUMBER() OVER (PARTITION BY "Customer ID" ORDER BY "Order Date") AS order_row_number
FROM superstore_raw
ORDER BY customer_id, order_row_number;
"""

ranked_orders_per_customer_df = pd.read_sql_query(query_row_number_per_customer, conn)
display(ranked_orders_per_customer_df)
print(f"Total orders with row numbers: {len(ranked_orders_per_customer_df)}")

### Display Top 3 Customers Based on Total Sales (Window Function)

This query re-uses the CTE to calculate `CustomerTotalSales` and then applies the `RANK()` window function. A subquery is then used to filter for customers with a `sales_rank` of 3 or less, effectively showing the top 3 customers.

In [ ]:
query_top_3_customers = """
WITH CustomerTotalSales AS (
    SELECT
        "Customer ID" AS customer_id,
        SUM(Sales) AS total_sales
    FROM superstore_raw
    GROUP BY "Customer ID"
),
RankedCustomers AS (
    SELECT
        customer_id,
        total_sales,
        RANK() OVER (ORDER BY total_sales DESC) AS sales_rank
    FROM CustomerTotalSales
)
SELECT
    customer_id,
    total_sales,
    sales_rank
FROM RankedCustomers
WHERE sales_rank <= 3
ORDER BY sales_rank;
"""

top_3_customers_df = pd.read_sql_query(query_top_3_customers, conn)
display(top_3_customers_df)
print(f"Total top customers displayed: {len(top_3_customers_df)}")

### Final Query: Customer Name, Total Sales, and Rank (JOIN + CTE + Window Function)

This query demonstrates the power of combining SQL features. It first calculates total sales for each customer using a CTE, then joins this result with the `customers` table to retrieve customer names, and finally applies a `RANK()` window function to order customers by their total sales.

In [ ]:
query_final_customer_ranking = """
WITH CustomerTotalSales AS (
    SELECT
        "Customer ID" AS customer_id,
        SUM(Sales) AS total_sales
    FROM superstore_raw
    GROUP BY "Customer ID"
)
SELECT
    c.customer_name,
    cts.total_sales,
    RANK() OVER (ORDER BY cts.total_sales DESC) AS sales_rank
FROM CustomerTotalSales cts
JOIN customers c ON cts.customer_id = c.customer_id
ORDER BY sales_rank;
"""

final_customer_ranking_df = pd.read_sql_query(query_final_customer_ranking, conn)
display(final_customer_ranking_df)
print(f"Total customers in final ranking: {len(final_customer_ranking_df)}")

### Investigating Duplicates in the `customers` table

Let's check if there are any duplicate `customer_id` or `customer_name` entries in the `customers` table, which could explain the higher row count and repeated customer names in the final ranking query.

In [ ]:
print("Checking for duplicate Customer IDs in the 'customers' table:")
duplicate_customer_id_query = """
SELECT customer_id, COUNT(*) as count
FROM customers
GROUP BY customer_id
HAVING count > 1;
"""
duplicate_customer_ids = pd.read_sql_query(duplicate_customer_id_query, conn)
display(duplicate_customer_ids)

print("\nChecking for duplicate Customer Names in the 'customers' table:")
duplicate_customer_name_query = """
SELECT customer_name, COUNT(*) as count
FROM customers
GROUP BY customer_name
HAVING count > 1;
"""
duplicate_customer_names = pd.read_sql_query(duplicate_customer_name_query, conn)
display(duplicate_customer_names)